# Capítulo 13 — Escalas, aproximaciones y perturbaciones

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Qué término domina? La pregunta que resuelve medio problema.

Ecuación cuadrática con un parámetro pequeño: la perturbación regular encuentra
una raíz y pierde la otra. El balance dominante la recupera.

La figura responde: ¿cómo se detecta que una perturbación es singular?

Ejecutar:  python fig_balance_dominante.py

*(script original: `codigo/fig_balance_dominante.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

eps = np.logspace(-6, -0.3, 300)

# eps x^2 + x - 1 = 0  -> raíces exactas
raiz_pos = (-1 + np.sqrt(1 + 4 * eps)) / (2 * eps)
raiz_neg = (-1 - np.sqrt(1 + 4 * eps)) / (2 * eps)

# Perturbación regular: x = x0 + eps x1 + ...
regular = 1 - eps + 2 * eps**2

# Reescalado singular: x = X/eps  ->  X^2 + X - eps = 0  ->  X ~ -1
singular = -1 / eps - 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

ax1.semilogx(eps, raiz_pos, color=C.ink, lw=2.4, label="raíz exacta 1")
ax1.semilogx(eps, regular, "--", color=C.blue, lw=1.8,
             label=r"regular: $1-\epsilon+2\epsilon^2$")
ax1.set_xlabel(r"$\epsilon$"), ax1.set_ylabel("raíz")
ax1.set_title(r"$\epsilon x^2+x-1=0$: la raíz que sí se ve")
ax1.legend(fontsize=8.5)
ax1.set_ylim(0.6, 1.05)

ax2.loglog(eps, -raiz_neg, color=C.ink, lw=2.4, label="raíz exacta 2")
ax2.loglog(eps, -singular, "--", color=C.red, lw=1.8,
           label=r"reescalando $x=X/\epsilon$:  $-1/\epsilon-1$")
ax2.set_xlabel(r"$\epsilon$"), ax2.set_ylabel("$-$raíz")
ax2.set_title("La raíz que la perturbación regular pierde")
ax2.legend(fontsize=8.5)
ax2.annotate(r"se escapa a $\infty$ cuando $\epsilon\to0$",
             xy=(1e-5, 1e5), xytext=(3e-4, 3e2), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

for e in (1e-2, 1e-4):
    exactas = np.roots([e, 1, -1])
    print(f"eps={e:.0e}: raíces exactas = {np.sort(exactas)}, "
          f"regular = {1-e+2*e**2:.6f}, singular = {-1/e-1:.1f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Cuando el término pequeño manda: capas límite.

Problema eps y'' + y' + y = 0 con y(0)=0, y(1)=1: solución exacta, solución
exterior, solución interior y su empalme.

La figura responde: ¿por qué despreciar el término con el parámetro pequeño
puede ser catastrófico?

Ejecutar:  python fig_capa_limite.py

*(script original: `codigo/fig_capa_limite.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

x = np.linspace(0, 1, 3000)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

for eps, color in zip([0.2, 0.05, 0.01], [C.ochre, C.green, C.blue]):
    # raíces de eps m^2 + m + 1 = 0
    disc = np.sqrt(1 - 4 * eps)
    m1 = (-1 + disc) / (2 * eps)          # lenta, ~ -1
    m2 = (-1 - disc) / (2 * eps)          # rápida, ~ -1/eps
    A = 1 / (np.exp(m1) - np.exp(m2))
    y = A * (np.exp(m1 * x) - np.exp(m2 * x))
    ax1.plot(x, y, color=color, lw=1.8, label=f"$\\epsilon$ = {eps}")
    print(f"eps={eps}: m1={m1:.3f}, m2={m2:.1f}, anchura de capa ~ {eps:.3f}")

# Solución exterior (ignorando el término eps y''): y' + y = 0 con y(1)=1
ax1.plot(x, np.exp(1 - x), "--", color=C.red, lw=2.0,
         label=r"exterior: $e^{1-x}$")
ax1.plot(0, np.e, "o", color=C.red, ms=7)
ax1.annotate("la exterior no cumple\n$y(0)=0$", xy=(0, np.e),
             xytext=(0.25, 2.3), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
ax1.set_xlabel("$x$"), ax1.set_ylabel("$y$")
ax1.set_title("Despreciar el término pequeño rompe la condición de contorno")
ax1.legend(fontsize=8, loc="upper right")

# --- Empalme --------------------------------------------------------------
eps = 0.02
disc = np.sqrt(1 - 4 * eps)
m1, m2 = (-1 + disc) / (2 * eps), (-1 - disc) / (2 * eps)
A = 1 / (np.exp(m1) - np.exp(m2))
exacta = A * (np.exp(m1 * x) - np.exp(m2 * x))
exterior = np.exp(1 - x)
interior = np.e * (1 - np.exp(-x / eps))
compuesta = exterior + np.e * (-np.exp(-x / eps))

ax2.plot(x, exacta, color=C.ink, lw=2.6, label="exacta", alpha=0.9)
ax2.plot(x, exterior, "--", color=C.red, lw=1.5, label="exterior")
ax2.plot(x, interior, ":", color=C.blue, lw=2.0, label="interior")
ax2.plot(x, compuesta, "-", color=C.green, lw=1.5, label="compuesta")
ax2.axvspan(0, 5 * eps, color=C.grey, alpha=0.18)
ax2.text(5 * eps + 0.02, 0.5, f"capa límite\nde anchura $\\epsilon$ = {eps}",
         fontsize=8.4, color=C.ink)
ax2.set_xlabel("$x$"), ax2.set_ylabel("$y$")
ax2.set_title(r"Empalme asintótico ($\epsilon = 0{,}02$)")
ax2.legend(fontsize=8, loc="lower right")
print(f"error máximo de la compuesta: {np.abs(compuesta-exacta).max():.4f}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Una serie que diverge y aun así es la mejor herramienta que tienes.

Serie asintótica de la integral exponencial: el error baja, alcanza un mínimo
y después crece sin límite.

La figura responde: ¿cuántos términos hay que sumar de una serie divergente?

Ejecutar:  python fig_serie_asintotica.py

*(script original: `codigo/fig_serie_asintotica.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.special import exp1

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# f(x) = e^x E1(x) ~ sum_{n>=0} (-1)^n n! / x^{n+1}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

for x0, color in zip([3.0, 5.0, 10.0], [C.red, C.ochre, C.blue]):
    exacto = np.exp(x0) * exp1(x0)
    parciales, errores = [], []
    s = 0.0
    for n in range(0, 26):
        s += (-1)**n * math.factorial(n) / x0**(n + 1)
        parciales.append(s)
        errores.append(abs(s - exacto))
    ax1.semilogy(range(len(errores)), errores, "o-", color=color, ms=4, lw=1.3,
                 label=f"$x$ = {x0:.0f}")
    n_opt = int(np.argmin(errores))
    ax1.plot(n_opt, errores[n_opt], "*", color=color, ms=15, zorder=5)
    print(f"x={x0}: mejor con {n_opt} términos, error {errores[n_opt]:.2e}; "
          f"con 25 términos, error {errores[-1]:.2e}")

ax1.set_xlabel("términos sumados $N$"), ax1.set_ylabel("error absoluto")
ax1.set_title("La serie diverge: el error tiene un mínimo")
ax1.legend(fontsize=8.5)

# --- Precisión óptima frente a x -----------------------------------------
xs = np.linspace(2, 25, 60)
mejores, n_opts = [], []
for x0 in xs:
    exacto = np.exp(x0) * exp1(x0)
    s, err = 0.0, []
    for n in range(0, 60):
        s += (-1)**n * math.factorial(n) / x0**(n + 1)
        err.append(abs(s - exacto))
    mejores.append(min(err)), n_opts.append(int(np.argmin(err)))

ax2.semilogy(xs, mejores, color=C.blue, lw=2.0, label="mejor error alcanzable")
ax2.semilogy(xs, np.exp(-xs) * np.sqrt(2 * np.pi / xs), "--", color=C.ink,
             lw=1.5, label=r"$\sim e^{-x}$")
ax2b = ax2.twinx()
ax2b.plot(xs, n_opts, color=C.red, lw=1.5)
ax2b.set_ylabel("términos óptimos", color=C.red)
ax2b.tick_params(axis="y", colors=C.red)
ax2b.grid(False)
ax2.set_xlabel("$x$"), ax2.set_ylabel("error mínimo")
ax2.set_title(r"El error óptimo decae como $e^{-x}$")
ax2.legend(fontsize=8.5, loc="upper right")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Una aproximación no es buena o mala: es buena hasta cierto punto. ¿Cuál?

Desarrollos de Taylor de sin(x) y de 1/(1+x) con su error, y el radio de
validez para una tolerancia dada.

La figura responde: ¿hasta dónde puedo usar «para ángulos pequeños»?

Ejecutar:  python fig_taylor_validez.py

*(script original: `codigo/fig_taylor_validez.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import math
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

x = np.linspace(0, 3.0, 800)
ordenes = [(1, r"$x$"), (3, r"$x-\frac{x^3}{6}$"),
           (5, r"$x-\frac{x^3}{6}+\frac{x^5}{120}$")]
colores = [C.red, C.ochre, C.green]

for (n, etiqueta), color in zip(ordenes, colores):
    aprox = sum((-1)**k * x**(2 * k + 1) / math.factorial(2 * k + 1)
                for k in range((n + 1) // 2))
    err = np.abs(aprox - np.sin(x)) / np.maximum(np.abs(np.sin(x)), 1e-12)
    ax1.semilogy(x, np.maximum(err, 1e-17), color=color, lw=1.8, label=etiqueta)
    for tol in (0.01,):
        idx = np.argmax(err > tol)
        if idx:
            ax1.plot(x[idx], tol, "o", color=color, ms=6)
            print(f"sin(x) con {etiqueta}: error < 1 % hasta x = {x[idx]:.3f} rad "
                  f"= {np.degrees(x[idx]):.0f}°")

ax1.axhline(0.01, color=C.ink, ls="--", lw=1.2)
ax1.text(0.05, 0.013, "tolerancia del 1 %", fontsize=8.4, color=C.ink)
ax1.set_xlabel("$x$ (rad)"), ax1.set_ylabel("error relativo")
ax1.set_title(r"$\sin x$: ¿hasta dónde vale «ángulo pequeño»?")
ax1.legend(fontsize=9, loc="lower right")
ax1.set_ylim(1e-14, 2)

# --- Serie con radio de convergencia finito ------------------------------
x2 = np.linspace(0, 1.6, 800)
for n, color in zip([2, 5, 10, 30], [C.red, C.ochre, C.green, C.blue]):
    aprox = sum((-x2)**k for k in range(n + 1))
    err = np.abs(aprox - 1 / (1 + x2))
    ax2.semilogy(x2, np.maximum(err, 1e-17), color=color, lw=1.5,
                 label=f"{n} términos")
ax2.axvline(1.0, color=C.ink, lw=1.6)
ax2.text(1.03, 1e-10, "radio de\nconvergencia", fontsize=8.4, color=C.ink)
ax2.set_xlabel("$x$"), ax2.set_ylabel("error absoluto")
ax2.set_title(r"$1/(1+x)$: más términos no siempre ayudan")
ax2.legend(fontsize=8, loc="lower right")
ax2.set_ylim(1e-14, 1e6)
ax2.annotate("aquí, añadir términos\nempeora el resultado",
             xy=(1.35, 1e3), xytext=(0.15, 1e3), fontsize=8.4, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
